### 1. Environment Setup
First, ensure the Ultralytics package is installed and check for GPU acceleration.

In [1]:
# Install the ultralytics package
!pip install ultralytics -q

import os
from ultralytics import YOLO

# Verify GPU availability
import torch
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda


### 2. Fine-Tuning the Model
When using Kaggle, you will typically point to a data.yaml file provided by the dataset you've added to your notebook.

In [2]:
# 1. Load a pretrained YOLOv8 Nano model (best for speed and learning)
model = YOLO('yolov8n.pt') 

# 2. Define your dataset path 
# Replace 'dataset-name' with the actual folder name in your /kaggle/input/
dataset_yaml = '/kaggle/input/datasets/barkataliarbab/license-plate-detection-dataset-10125-images/data.yaml'

# 3. Fine-tune the model
# Training for 50 epochs is usually sufficient for license plates
results = model.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolov8_license_plates',
    device=0 # Uses the first GPU
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/barkataliarbab/license-plate-detection-dataset-10125-images/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_license_plates, nbs=64, nms=False, opset=None, o

In [7]:
# 1. Export the model to ONNX format (optional, but good for deployment)
from ultralytics import YOLO
path_to_best = '/kaggle/working/runs/detect/yolov8_license_plates/weights/best.pt'
trained_model = YOLO(path_to_best)
trained_model.export(format='onnx')

print(f"Model saved and exported. You can find 'best.pt' and 'best.onnx' in /kaggle/working/")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/yolov8_license_plates/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 262ms
Prepared 2 packages in 2.62s
Installed 2 packages in 13ms
 + onnxruntime-gpu==1.24.3
 + onnxslim==0.1.87

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 4.9s, saved as '/kaggle/working/runs/detect/yolov8_license_plates/weights/best.onnx' (11.7 MB)

Export complete (5.3s)
Results saved to /kaggle/working/runs/detect/yolov8_license_plates/weights
Predict:         yolo predict task=detect model=/kaggle/working/runs/detect/yolov8_license_plates/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/detect/yolov8_license_plates/weights/best.onnx imgsz=640 data=/kaggle/input/datasets/barkataliarbab/license-plate-detection-dataset-10125-images/data.yaml  
Visualize:       https://netron.app
Model saved and exported. You can find 'best.pt' and 'best.onnx' in /kaggle/working/


In [10]:
import cv2
import csv
from ultralytics import YOLO

# --- CONFIGURATION ---
video_input_path = '/kaggle/input/datasets/pranaynyachhyon/license-plate-output1/license_plate_output.mp4' 
video_output_path = '/kaggle/working/license_plate_output.mp4'
csv_output_path = '/kaggle/working/detection_log.csv'

# Load your fine-tuned model
best_model = YOLO("/kaggle/working/runs/detect/yolov8_license_plates/weights/best.pt")

# Open the video source
cap = cv2.VideoCapture(video_input_path)
if not cap.isOpened():
    print("Error: Could not open video.")
else:
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_output_path, fourcc, fps, (width, height))

    # --- CSV SETUP ---
    # Open CSV file and write the header
    csv_file = open(csv_output_path, mode='w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Frame', 'Object_ID', 'Class', 'Confidence', 'X1', 'Y1', 'X2', 'Y2'])

    print("Processing video and logging data... this may take a few minutes.")

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run inference (stream=True is better for video memory)
        results = best_model.predict(frame, conf=0.5, device=0, verbose=False)

        # Plot for the video output
        annotated_frame = results[0].plot()
        out.write(annotated_frame)

        # --- CSV LOGGING LOGIC ---
        for result in results:
            boxes = result.boxes
            for i, box in enumerate(boxes):
                # Extract data
                coords = box.xyxy[0].tolist() # [x1, y1, x2, y2]
                conf = box.conf[0].item()
                cls = result.names[int(box.cls[0])]

                # Write row to CSV
                csv_writer.writerow([
                    frame_count, 
                    i, 
                    cls, 
                    f"{conf:.2f}", 
                    int(coords[0]), int(coords[1]), int(coords[2]), int(coords[3])
                ])

        frame_count += 1

    # Release everything
    csv_file.close()
    cap.release()
    out.release()
    print(f"Complete! Video saved to: {video_output_path}")
    print(f"Data log saved to: {csv_output_path}")

Processing video and logging data... this may take a few minutes.
Complete! Video saved to: /kaggle/working/license_plate_output.mp4
Data log saved to: /kaggle/working/detection_log.csv


In [3]:
import cv2
from ultralytics import YOLO

# 1. Load your fine-tuned model (Ensure best.pt is in the same folder as this script)
model = YOLO("/kaggle/working/runs/detect/yolov8_license_plates/weights/best.pt")

# 2. Open the local webcam (0 is usually the built-in camera)
cap = cv2.VideoCapture(0)

# Check if the webcam is opened correctly
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

print("Press 'q' to quit the camera feed.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 3. Run YOLOv8 inference on the live frame
    # We use stream=True for better real-time performance
    results = model.predict(frame, conf=0.5, stream=True)

    for result in results:
        # Plot the bounding boxes and labels on the frame
        annotated_frame = result.plot()

        # 4. Display the resulting frame
        cv2.imshow("YOLOv8 Real-Time License Plate Detection", annotated_frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

Error: Could not open webcam.
Press 'q' to quit the camera feed.


[ WARN:0@117.791] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@117.792] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
